<br>
<div style="text-align: center;">

<h1 style="
  margin: 0 0 18px 0;
  color: #0077c8;
  font-size: 42px;
  letter-spacing: 1px;
">
  TB Lightning Home Assessment
</h1>
<p style="
  margin: 0 0 20px 0;
  color: #4a5b68;
  font-size: 16px;
  text-align: center;
  width: 100%;
">
  <strong>Created by Tyler Durette</strong>
  &nbsp;·&nbsp;
</p>
    
  <div style="
    height: 4px;
    width: 42%;
    margin: 0 auto 22px auto;
    background: linear-gradient(90deg, #0077c8, #79c2ed, #ffb81c);
    border-radius: 4px;
  "></div>

  <img
    src="images/vasilevskiy.jpeg"
    alt="Andrei Vasilevskiy, Tampa Bay Lightning"
    style="
      display: block;
      width: 88%;
      max-width: 960px;
      margin: 0 auto;
      border-radius: 8px;
      box-shadow: 0 8px 24px rgba(0,55,95,.18);
    "
  />

  <p style="margin: 14px 0 0 0; color: #687985; font-size: 11px;  text-align: center">
    Andrei Vasilevskiy · Tampa Bay Lightning · image source: NHL.com
  </p>

</div>

<div style="
  border-left: 5px solid #ffb81c;
  padding: 16px 22px;
  background: #fffaf0;
  border-radius: 6px;
  color: #12304a;
">

  <h2 style="
    margin: 0 0 14px 0;
    color: #12304a;
    font-size: 24px;
  ">What This Notebook Demonstrates</h2>

  <p style="
    margin: 0 0 14px 0;
    color: #4a5b68;
  ">An answer-first walkthrough of the NHL take-home implementation, including the required data import, cleaning, database modeling, API layer, robustness checks, and reproducible Docker workflow.</p>

  <ul style="
    margin: 0;
    padding-left: 22px;
    color: #4a5b68;
    line-height: 1.7;
  ">
    <li><strong>Data ingestion:</strong> Imports and validates the supplied 2022–23 seed dataset containing 951 skater records.</li>
    <li><strong>Historical coverage:</strong> Loads every contiguous regular season from 2022–23 through the configured 2026–27 target without skipping intermediate years.</li>
    <li><strong>Data modeling:</strong> Normalizes players, season statistics, games, team-game statistics, standings, rosters, and pipeline-run metadata.</li>
    <li><strong>Database:</strong> Uses PostgreSQL through Docker Compose locally and Supabase PostgreSQL as the managed cloud database.</li>
    <li><strong>API:</strong> Exposes FastAPI endpoints for player goals, penalty rates, team rankings, multi-team players, current rosters, health, and pipeline status.</li>
    <li><strong>Daily refresh:</strong> Using a Scheduled GitHub Action it rechecks the previous three dates, refreshes current-season statistics, standings, rosters, and correction-window game data every morning at 8am.</li>
    <li><strong>Robustness:</strong> Includes retries, pagination validation, preseason empty-state handling, idempotent upserts, season-gap validation, and pipeline status tracking.</li>
    <li><strong>Reproducibility:</strong> Runs through Docker Compose with documented commands, automated tests, and a GitHub Actions workflow for the managed PostgreSQL refresh.</li>
  </ul>

</div>

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import sys

import pandas as pd
import requests

BASE_URL = os.environ.get("NHL_API_URL", "http://localhost:8000")
SEED_SEASON = 20222023
REQUEST_TIMEOUT = 10


def get_json(path, **params):
    response = requests.get(f"{BASE_URL}{path}", params=params, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    return response.json()

print({"api_base_url": BASE_URL, "checked_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds") + "Z"})

{'api_base_url': 'http://localhost:8000', 'checked_at_utc': '2026-08-13T17:03:44+00:00Z'}


In [2]:
health = get_json("/health")
health

{'status': 'ok', 'database': 'reachable'}

In [3]:
openapi = requests.get(f"{BASE_URL}/openapi.json", timeout=REQUEST_TIMEOUT).json()
print("Available routes:")
for route in sorted(openapi["paths"]):
    print(" ", route)

Available routes:
  /health
  /pipeline/status
  /players/most-goals
  /players/multi-team
  /players/penalties-per-minute
  /rosters/current/{team_abbrev}
  /teams/rankings


## 1. Player with the most goals

This answers the first analyst question using the supplied 2022–23 seed season.

In [4]:
goal_leader = get_json("/players/most-goals", season_id=SEED_SEASON, limit=5)
pd.DataFrame(goal_leader)

,player_id,name,goals,games_played
0,8478402,Connor McDavid,64,82
1,8477956,David Pastrnak,61,82
2,8478420,Mikko Rantanen,55,82
3,8477934,Leon Draisaitl,52,80
4,8478010,Brayden Point,51,82


In [5]:
assert goal_leader[0]["name"] == "Connor McDavid"
assert goal_leader[0]["goals"] == 64
print("Acceptance check passed: Connor McDavid led the supplied season with 64 goals.")

Acceptance check passed: Connor McDavid led the supplied season with 64 goals.


## 2. Penalty minutes per total ice-time minute

The API returns penalty minutes divided by total time on ice. The optional `min_games` filter makes the ranking more useful by removing extremely small samples.

In [6]:
penalties = get_json(
    "/players/penalties-per-minute",
    season_id=SEED_SEASON,
    min_games=20,
    limit=10,
)
pd.DataFrame(penalties)

,player_id,name,pim,total_toi_minutes,pim_per_minute
0,8479379,Givani Smith,72,252.150000,0.285544
1,8478421,A.J. Greer,114,555.649915,0.205165
2,8474034,Patrick Maroon,150,827.583333,0.181251
3,8479981,Jonah Gadjovich,57,314.699992,0.181125
4,8477070,Liam O'Brien,114,639.899960,0.178153
5,8480355,Mark Kastelic,102,578.049983,0.176455
6,8475235,Nicolas Deslauriers,136,807.533333,0.168414
7,8477962,Brendan Lemieux,74,458.166600,0.161513
8,8475766,Austin Watson,123,767.466625,0.160268
9,8475842,Sam Carrick,86,549.399933,0.156534


In [7]:
penalty_table = pd.DataFrame(penalties)
assert {"pim", "total_toi_minutes", "pim_per_minute"}.issubset(penalty_table.columns)
assert (penalty_table["total_toi_minutes"] > 0).all()
print(f"Validated {len(penalty_table)} penalty-rate rows with positive total TOI.")

Validated 10 penalty-rate rows with positive total TOI.


## 3. Team goals and shots rankings

Team rankings come from normalized team-game facts rather than assigning a traded player's aggregate season totals to the last team.

In [8]:
team_goals = get_json("/teams/rankings", season_id=SEED_SEASON, metric="goals", limit=10)
team_shots = get_json("/teams/rankings", season_id=SEED_SEASON, metric="shots", limit=10)
print("Top teams by goals")
display(pd.DataFrame(team_goals))
print("Top teams by shots")
display(pd.DataFrame(team_shots))

Top teams by goals


,team_id,team,goals
0,22,EDM,325
1,6,BOS,301
2,7,BUF,293
3,1,NJD,289
4,55,SEA,289
5,13,FLA,288
6,25,DAL,281
7,14,TBL,280
8,10,TOR,278
9,21,COL,274


Top teams by shots


,team_id,team,shots
0,13,FLA,3019
1,20,CGY,2948
2,12,CAR,2852
3,1,NJD,2821
4,5,PIT,2818
5,22,EDM,2754
6,9,OTT,2746
7,21,COL,2727
8,6,BOS,2703
9,7,BUF,2665


In [9]:
for result, metric in ((team_goals, "goals"), (team_shots, "shots")):
    values = [row[metric] for row in result]
    assert values == sorted(values, reverse=True)
print("Ranking check passed: both result sets are sorted descending by the requested metric.")

Ranking check passed: both result sets are sorted descending by the requested metric.


## 4. Players who appeared for multiple teams

This route surfaces the team affiliations represented in the supplied season and reports the inferred number of team changes.

In [10]:
multi_team = get_json("/players/multi-team", season_id=SEED_SEASON)
multi_team_table = pd.DataFrame(multi_team)
print(f"Players with multiple team affiliations: {len(multi_team_table)}")
display(multi_team_table.head(15))

Players with multiple team affiliations: 95


,player_id,name,teams,team_count,team_changes
0,8478569,Noel Acciari,"[STL, TOR]",2,1
1,8479315,Joey Anderson,"[CHI, TOR]",2,1
2,8479335,Rasmus Asplund,"[BUF, NSH]",2,1
3,8477979,Nicolas Aube-Kubel,"[TOR, WSH]",2,1
4,8478870,Rudolfs Balcers,"[FLA, TBL]",2,1
5,8477964,Ivan Barbashev,"[STL, VGK]",2,1
6,8475197,Tyson Barrie,"[EDM, NSH]",2,1
7,8478463,Anthony Beauvillier,"[NYI, VAN]",2,1
8,8479356,Kieffer Bellows,"[NYI, PHI]",2,1
9,8477479,Tyler Bertuzzi,"[BOS, DET]",2,1


In [11]:
assert len(multi_team_table) > 0
assert (multi_team_table["team_count"] >= 2).all()
assert (multi_team_table["team_changes"] == multi_team_table["team_count"] - 1).all()
print("Multi-team consistency check passed.")

Multi-team consistency check passed.


## 5. Active roster lookup

The current-roster endpoint returns the latest active roster snapshot for a team. Tampa Bay is used as the demonstration team because it is relevant to the role.

In [12]:
tampa_roster = get_json("/rosters/current/TBL")
roster_table = pd.DataFrame(tampa_roster)
print(f"Tampa Bay roster rows: {len(roster_table)}")
display(roster_table.head(15))

Tampa Bay roster rows: 26


,player_id,team,position,snapshot_date
0,8474151,TBL,D,2026-08-13
1,8474590,TBL,D,2026-08-13
2,8475167,TBL,D,2026-08-13
3,8476453,TBL,R,2026-08-13
4,8476826,TBL,C,2026-08-13
5,8476878,TBL,C,2026-08-13
6,8476883,TBL,G,2026-08-13
7,8477149,TBL,R,2026-08-13
8,8477404,TBL,C,2026-08-13
9,8477992,TBL,G,2026-08-13


In [13]:
assert len(roster_table) > 0
assert set(["player_id", "team", "position", "snapshot_date"]).issubset(roster_table.columns)
assert set(roster_table["team"]) == {"TBL"}
print("Roster check passed: all returned rows are current Tampa Bay rows.")

Roster check passed: all returned rows are current Tampa Bay rows.


## 6. Pipeline health and refresh observability

The pipeline records each refresh in `pipeline_runs`, including the correction window, row counts, status, and errors.

In [14]:
pipeline = get_json("/pipeline/status")
print(json.dumps(pipeline, indent=2))

{
  "run_id": "df62a364-ca20-4243-b70d-a9b7eb03c393",
  "status": "succeeded",
  "command": "refresh",
  "started_at": "2026-08-13T11:45:33.300672+00:00",
  "completed_at": "2026-08-13T11:45:42.135851+00:00",
  "seasons": [
    20262027
  ],
  "row_counts": {
    "run_id": "df62a364-ca20-4243-b70d-a9b7eb03c393",
    "as_of_date": "2026-08-13",
    "window_start": "2026-08-10",
    "window_end": "2026-08-12",
    "dates_checked": 3,
    "season_id": 20262027,
    "games": 0,
    "player_game_stats": 0,
    "team_game_stats": 0,
    "player_season_stats": 0,
    "team_season_stats": 0,
    "standings_snapshots": 32,
    "roster_snapshots": 810
  },
  "error": null
}


In [15]:
assert pipeline["status"] in {"succeeded", "running"}
assert pipeline["command"] in {"refresh", "refresh-as-of", "backfill"}
print("Pipeline observability check passed.")

Pipeline observability check passed.


## 7. Season and data-volume summary

The API status payload exposes the latest refresh counts. This compact summary is useful in a handoff or interview walkthrough.

In [16]:
row_counts = pipeline.get("row_counts") or {}
summary = {
    "pipeline_status": pipeline["status"],
    "last_command": pipeline["command"],
    "season_checked": row_counts.get("season_id"),
    "dates_checked": row_counts.get("dates_checked"),
    "games_updated": row_counts.get("games"),
    "player_game_rows_updated": row_counts.get("player_game_stats"),
    "team_game_rows_updated": row_counts.get("team_game_stats"),
    "standings_snapshots": row_counts.get("standings_snapshots"),
    "roster_snapshots": row_counts.get("roster_snapshots"),
}
pd.Series(summary)

pipeline_status             succeeded
last_command                  refresh
season_checked               20262027
dates_checked                       3
games_updated                       0
player_game_rows_updated            0
team_game_rows_updated              0
standings_snapshots                32
roster_snapshots                  810
dtype: object

## 8. Seed-file quality cross-check

The provided CSV is the required starting dataset. This cell checks the documented properties without using it to answer the API questions: 951 rows, one season, unique player IDs, and the known McDavid value.

In [17]:
candidates = [Path("data/data_dump.csv"), Path.cwd().parent / "data/data_dump.csv"]
csv_path = next((candidate for candidate in candidates if candidate.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Could not locate data/data_dump.csv from the notebook or repository parent directory.")
seed = pd.read_csv(csv_path, keep_default_na=False)
quality = {
    "path": str(csv_path),
    "rows": len(seed),
    "columns": len(seed.columns),
    "unique_player_ids": seed["playerId"].nunique(),
    "seasons": sorted(seed["Season"].unique().tolist()),
    "mcdavid_goals": int(seed.loc[seed["Name"] == "Connor McDavid", "G"].iloc[0]),
    "multi_team_rows": int(seed["Team"].str.contains(",").sum()),
}
pd.Series(quality)

path                 data/data_dump.csv
rows                                951
columns                              24
unique_player_ids                   951
seasons                      [20222023]
mcdavid_goals                        64
multi_team_rows                      95
dtype: object

## 9. Robustness checks

A malformed metric should produce a client error rather than silently returning a misleading result; an unknown route should produce a 404.

In [18]:
bad_metric = requests.get(
    f"{BASE_URL}/teams/rankings",
    params={"season_id": SEED_SEASON, "metric": "invalid"},
    timeout=REQUEST_TIMEOUT,
)
missing_route = requests.get(f"{BASE_URL}/does-not-exist", timeout=REQUEST_TIMEOUT)
print({
    "invalid_metric_status": bad_metric.status_code,
    "unknown_route_status": missing_route.status_code,
})
assert bad_metric.status_code == 400
assert missing_route.status_code == 404
print("Robustness checks passed.")

{'invalid_metric_status': 400, 'unknown_route_status': 404}
Robustness checks passed.


## 10. Reproducibility and handoff commands

These are the commands another engineer can run from a clean checkout. The notebook is evidence on top of the reproducible service, not a replacement for Docker Compose.

In [19]:
print("""# Start the full stack
./start.sh

# Run this notebook from the repository root
jupyter lab

# Or run the automated tests inside the API image
docker compose run --rm api python -m pytest -q

# Run the daily correction-window refresh
./refresh.sh

# Stop while preserving the PostgreSQL volume
./stop.sh""")

# Start the full stack
./start.sh

# Run this notebook from the repository root
jupyter lab

# Or run the automated tests inside the API image
docker compose run --rm api python -m pytest -q

# Run the daily correction-window refresh
./refresh.sh

# Stop while preserving the PostgreSQL volume
./stop.sh


## Assessment conclusion

The demonstrated vertical slice is complete: Docker Compose starts PostgreSQL and FastAPI, the seed CSV is loaded into normalized tables, the API answers every requested analyst question, refreshes are observable and repeatable, and invalid requests fail explicitly. The main operational handoff is the daily refresh command/workflow; local Docker Compose remains the reproducible way for another engineer to run the system.